# Parkinson Speech AI - Colab run

Runs the pipeline end to end: quality gate -> phonation + DDK models -> fusion -> screening score.

Setup needed before running:
- Upload `italian_pvs.zip` to Drive at `MyDrive/parkinson-speech-ai-data/italian_pvs.zip` (too big for GitHub)
- `26_29_09_2017_KCL.zip` too if you want the ReadText baseline as well
- Set `REPO_URL` and branch in the clone cell below

Models are already committed to the repo so training isn't required to run this, that part is optional.

In [ ]:
REPO_URL = "https://github.com/prasad-xma/KND_CDAP_SE_01.git"
BRANCH = "component/matheesha"
REPO_DIR = "KND_CDAP_SE_01"

## Mount drive, clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
%cd /content
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}/parkinson-speech-ai

## Install requirements

In [ ]:
!pip install -q -r requirements.txt
!pip install -q noisereduce pystoi pesq

## Copy datasets from Drive

In [ ]:
import shutil

DRIVE_DATA_DIR = "/content/drive/MyDrive/parkinson-speech-ai-data"
os.makedirs("data/italian_pvs", exist_ok=True)

italian_src = os.path.join(DRIVE_DATA_DIR, "italian_pvs.zip")
if os.path.exists(italian_src):
    shutil.copy(italian_src, "data/italian_pvs/italian_pvs.zip")
    print("copied italian_pvs.zip")
else:
    print(f"missing {italian_src} on drive - component 2/3 italian baselines and fusion need this")

kcl_src = os.path.join(DRIVE_DATA_DIR, "26_29_09_2017_KCL.zip")
if os.path.exists(kcl_src):
    shutil.copy(kcl_src, "data/26_29_09_2017_KCL.zip")
    print("copied 26_29_09_2017_KCL.zip")
else:
    print("26_29_09_2017_KCL.zip not on drive, skipping readtext baseline")

## Training (optional, models already in repo)

In [ ]:
!python -m src.component2_phonation.train_vowel
!python -m src.component2_phonation.train_vowel_combined
!python -m src.component2_phonation.train_sakar
!python -m src.component3_ddk.train
!python -m src.fusion.train_italian_pvs_fusion

## Component 1 - quality gate

SNR based for DDK recordings (real pauses to measure from), HNR based for sustained vowel (no pauses).

In [ ]:
import glob
from src.component1_robustness.quality_gate import check_quality

vowel_sample = sorted(glob.glob("data/vowel_task/raw/HC_AH/*.wav"))[0]
ddk_sample = sorted(glob.glob("data/italian_pvs/raw/22 Elderly Healthy Control/*/D1*.wav"))[0]

print("vowel:", check_quality(vowel_sample))
print("ddk  :", check_quality(ddk_sample))

## Component 1 - enhancement check

Adds noise to a clean recording then runs it through the enhancement step, before/after. Enhancement raises the SNR/HNR estimate but currently lowers Component 2's prediction agreement with the clean version (see results/component1_robustness/enhancement_summary.csv) - so this isn't wired into the main pipeline by default, the gate just flags bad recordings for re-recording instead.

In [ ]:
import librosa, soundfile as sf
from src.common.audio_io import ensure_wav
from src.component1_robustness.degradation import add_white_noise
from src.component1_robustness.enhancement import enhance_audio

wav = ensure_wav(vowel_sample)
y, sr = librosa.load(wav, sr=None, mono=True)
noisy = add_white_noise(y, snr_db=5)
sf.write("noisy.wav", noisy, sr)
enhance_audio("noisy.wav", "enhanced.wav")

print("noisy   :", check_quality("noisy.wav"))
print("enhanced:", check_quality("enhanced.wav"))

## Component 2 - phonation model

Best tuned result (feature selection + hyperparameter search on combined AH + Italian PVS vowel data), then a live prediction on one sample.

In [ ]:
import json
with open("results/component2_phonation/vowel_combined_tuned_metrics.json") as f:
    c2_metrics = json.load(f)

print("cv accuracy:", round(c2_metrics["cv_accuracy"], 4))
print("cv balanced accuracy:", round(c2_metrics["cv_balanced_accuracy"], 4))
print("best params:", c2_metrics["best_params"])

report = c2_metrics["classification_report"]
for label in ["HC", "PD"]:
    r = report[label]
    print(f"{label}: precision={r['precision']:.3f}  recall={r['recall']:.3f}  f1={r['f1-score']:.3f}")

In [ ]:
from src.component2_phonation.predict import predict

tuned_model_path = "models/component2_phonation_vowel_combined_tuned.joblib"
result_c2 = predict(vowel_sample, model_path=tuned_model_path)

print("audio:", result_c2["audio_path"])
print("prediction:", result_c2["prediction"])
print(f"P(HC)={result_c2['probability_hc']:.3f}  P(PD)={result_c2['probability_pd']:.3f}")

## Component 3 / fusion ablation

vowel-only vs ddk-only vs three ways of combining them, same CV split for all.

In [ ]:
import json
with open("results/fusion/italian_pvs_fusion_ablation_metrics.json") as f:
    fusion_metrics = json.load(f)

for strategy in ["vowel_only", "ddk_only", "decision_level", "feature_level", "stacked", "weighted_ensemble"]:
    m = fusion_metrics[strategy]
    print(f"{strategy:16s} accuracy={m['accuracy']:.3f}  balanced_accuracy={m['balanced_accuracy']:.3f}")

print("\nbest:", fusion_metrics["best_strategy_by_balanced_accuracy"])
print("n_participants:", fusion_metrics["n_participants"], "- small sample, treat as preliminary")

## End to end screening score

In [ ]:
from src.screening.predict_screening import run_screening, print_report

hc_vowel = sorted(glob.glob("data/vowel_task/raw/HC_AH/*.wav"))[0]
hc_ddk = sorted(glob.glob("data/italian_pvs/raw/22 Elderly Healthy Control/*/D1*.wav"))[0]
result = run_screening(hc_vowel, hc_ddk)
print_report(hc_vowel, hc_ddk, result)

In [ ]:
pd_vowel = sorted(glob.glob("data/vowel_task/raw/PD_AH/*.wav"))[0]
pd_ddk = sorted(glob.glob("data/italian_pvs/raw/28 People with Parkinson's disease/*/*/D1*.wav"))[0]
result = run_screening(pd_vowel, pd_ddk)
print_report(pd_vowel, pd_ddk, result)

## Score gauge

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_gauge(score, classification):
    fig, ax = plt.subplots(figsize=(5, 3), subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('W')
    ax.set_theta_direction(-1)
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    colors = ['#2ca02c', '#ff9f1c', '#d62728']
    bounds = [0, 0.4, 0.7, 1.0]
    for i in range(3):
        ax.barh(0, (bounds[i+1]-bounds[i]) * np.pi, left=bounds[i] * np.pi, height=1, color=colors[i], alpha=0.85)
    needle_angle = score * np.pi
    ax.plot([needle_angle, needle_angle], [0, 1], color='black', linewidth=3)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_ylim(0, 1)
    ax.grid(False)
    ax.set_title(f"PD Screening Score: {score:.0%}\n{classification}", pad=20, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_gauge(result.fused_score, result.classification)